# 04 · Selektor z podglądem a losowy wybór (MoNuSeg)

**Pytanie.** Czy wybieranie obrazów, na których student faktycznie wypada najgorzej, uczy go szybciej niż losowanie?

**Selektor z podglądem** (`oracle_peek`) po każdej rundzie segmentuje studentem całą pulę nieoznaczoną, porównuje maski
z maskami wyroczni i bierze 2 obrazy o najniższym PQ. Czyta etykiety puli, więc **nie jest metodą**, tylko górną granicą
dla selektorów typu „wybieraj trudne” (niepewność, system rozmyty).

**Porównanie sparowane:** dla seeda *k* oba przebiegi startują z tego samego `student_init.pth` i tych samych 2 obrazów,
z tymi samymi hiperparametrami treningu. Runda 2 jest więc tym samym eksperymentem powtórzonym dwa razy i pokazuje
poziom szumu między uruchomieniami.

Miara główna: PQ maski studenta względem maski wyroczni na walidacji (`val_oracle_pq`). Zbiór testowy nie jest używany.

Dane: `outputs/al_random/full` i `outputs/al_oracle_peek/full` (WCSS, `slurm/monuseg_random_al.sbatch`).

In [ ]:
SEEDS = [0, 1, 2, 3, 4, 5]
ROUNDS = [2, 4, 6, 8, 10, 12]  # rundy wspólne dla obu przebiegów
METRIC = "val_oracle_pq"

In [ ]:
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from histopathology_pnw.config import Settings
from histopathology_pnw.datasets import monuseg

OUT = Settings().outputs_dir


def load(selector: str) -> pd.DataFrame:
    frames = []
    for seed in SEEDS:
        df = pd.read_csv(OUT / f"al_{selector}" / "full" / f"seed{seed}" / "results.csv")
        frames.append(df[df["n_labeled"].isin(ROUNDS)].assign(seed=seed))
    return pd.concat(frames, ignore_index=True)


rand, peek = load("random"), load("oracle_peek")
missing = {s: sorted(set(ROUNDS) - set(peek.loc[peek["seed"] == s, "n_labeled"])) for s in SEEDS}
print("brakujące rundy przebiegu z podglądem:", {s: m for s, m in missing.items() if m} or "brak")


def wide(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    return df.pivot(index="seed", columns="n_labeled", values=metric)


r, p = wide(rand, METRIC), wide(peek, METRIC)
diff = p - r

## Poziom szumu: runda 2

W rundzie 2 oba przebiegi trenują ten sam model na tych samych 2 obrazach. Różnica to niedeterminizm treningu na GPU,
czyli najmniejsza różnica, jaką w ogóle da się przypisać selektorowi.

In [ ]:
noise = pd.DataFrame({"losowy": r[2], "podgląd": p[2], "różnica": diff[2]})
print(
    f"runda 2: średnia |różnica| = {diff[2].abs().mean():.4f}, największa = {diff[2].abs().max():.4f}"
)
noise.round(4)

## Krzywe uczenia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, w, color in (("losowy", r, "tab:gray"), ("z podglądem", p, "tab:red")):
    for seed in w.index:
        axes[0].plot(w.columns, w.loc[seed], color=color, alpha=0.25, lw=1)
    axes[0].errorbar(
        w.columns,
        w.mean(),
        yerr=w.std(ddof=1),
        color=color,
        lw=2,
        marker="o",
        capsize=3,
        label=name,
    )
axes[0].set(
    xlabel="liczba oznaczonych obrazów",
    ylabel="PQ studenta względem wyroczni (walidacja)",
    title="Średnia ± odch. std. po seedach; cienkie linie = seedy",
    xticks=ROUNDS,
)
axes[0].legend()

for seed in diff.index:
    axes[1].plot(diff.columns, diff.loc[seed], marker=".", alpha=0.6, label=f"seed {seed}")
axes[1].plot(diff.columns, diff.mean(), color="k", lw=2.5, marker="o", label="średnia")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].axhspan(
    -diff[2].abs().max(), diff[2].abs().max(), color="gray", alpha=0.15, label="szum (runda 2)"
)
axes[1].set(
    xlabel="liczba oznaczonych obrazów",
    ylabel="podgląd minus losowy (PQ)",
    title="Różnica na seed (porównanie sparowane)",
    xticks=ROUNDS,
)
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
fig_dir = OUT / "al_oracle_peek" / "analysis"
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "peek_vs_random.png", dpi=130, bbox_inches="tight")

## Różnice w rundach i ALC

Dla każdej rundy: średnia różnica po seedach, liczba seedów, w których podgląd był lepszy, test t dla par
(i test Wilcoxona; przy 6 seedach najmniejsze możliwe p dwustronne to 0,031).

ALC(2–12) = pole pod krzywą metodą trapezów, podzielone przez długość przedziału (czyli średnie PQ w oknie 2–12).

In [ ]:
def paired(d: pd.Series) -> dict:
    d = d.dropna()
    t = stats.ttest_1samp(d, 0.0)
    try:
        w = stats.wilcoxon(d).pvalue
    except ValueError:
        w = np.nan
    return {
        "średnia różnica": float(d.mean()),
        "odch. std.": float(d.std(ddof=1)),
        "seedy z podglądem lepszym": f"{int((d > 0).sum())}/{len(d)}",
        "p (t)": float(t.pvalue),
        "p (Wilcoxon)": float(w),
    }


def alc(w: pd.DataFrame) -> pd.Series:
    x = np.array(w.columns, dtype=float)
    return w.apply(lambda row: np.trapezoid(row.to_numpy(), x) / (x[-1] - x[0]), axis=1)


per_round = pd.DataFrame({n: paired(diff[n]) for n in ROUNDS}).T
per_round.insert(0, "losowy", r.mean())
per_round.insert(1, "podgląd", p.mean())
per_round.index.name = "n_labeled"
alc_tab = pd.DataFrame({"losowy": alc(r), "podgląd": alc(p)})
alc_tab["różnica"] = alc_tab["podgląd"] - alc_tab["losowy"]
print(
    "ALC(2-12):",
    {
        k: (round(v, 4) if isinstance(v, float) else v)
        for k, v in paired(alc_tab["różnica"]).items()
    },
)
print(
    f"ALC losowy {alc_tab['losowy'].mean():.4f} ± {alc_tab['losowy'].std(ddof=1):.4f}, "
    f"podgląd {alc_tab['podgląd'].mean():.4f} ± {alc_tab['podgląd'].std(ddof=1):.4f}"
)
per_round.to_csv(fig_dir / "per_round.csv")
alc_tab.to_csv(fig_dir / "alc.csv")
per_round

In [ ]:
alc_tab.round(4)

## To samo względem ground truth

Kontrola: czy wniosek jest taki sam, gdy studenta oceniamy względem GT MoNuSeg (`val_pq`), a nie względem wyroczni.

In [ ]:
r_gt, p_gt = wide(rand, "val_pq"), wide(peek, "val_pq")
d_gt = p_gt - r_gt
gt_tab = pd.DataFrame({n: paired(d_gt[n]) for n in ROUNDS}).T
gt_tab.insert(0, "losowy", r_gt.mean())
gt_tab.insert(1, "podgląd", p_gt.mean())
a_gt = alc(p_gt) - alc(r_gt)
print(
    "ALC(2-12) względem GT:",
    {k: (round(v, 4) if isinstance(v, float) else v) for k, v in paired(a_gt).items()},
)
gt_tab

## Co wybierał podgląd

`selection.csv` z każdej rundy: PQ studenta względem wyroczni dla każdego obrazu puli i to, które zostały wybrane.

In [ ]:
POOL = monuseg.load_train_val_split()["train"]
# narząd jest znany tylko dla części obrazów (PDF dołączony do MoNuSeg)
organs = monuseg.organs().set_index("image_id")["organ"].reindex(POOL).fillna("nieznany")
sel = pd.concat(
    [
        pd.read_csv(
            OUT / "al_oracle_peek" / "full" / f"seed{s}" / f"round_{n:02d}" / "selection.csv"
        ).assign(seed=s, n_labeled=n)
        for s in SEEDS
        for n in ROUNDS
        if (
            OUT / "al_oracle_peek" / "full" / f"seed{s}" / f"round_{n:02d}" / "selection.csv"
        ).is_file()
    ],
    ignore_index=True,
)
sel["organ"] = sel["image_id"].map(organs)
sel.to_csv(fig_dir / "selection_all.csv", index=False)

pool_pq = sel.groupby("n_labeled").apply(
    lambda g: pd.Series(
        {
            "PQ puli: średnia": g["pq_oracle"].mean(),
            "PQ puli: min": g["pq_oracle"].min(),
            "PQ puli: odch. std.": g["pq_oracle"].std(ddof=1),
            "PQ wybranych": g.loc[g["selected"], "pq_oracle"].mean(),
        }
    ),
    include_groups=False,
)
pool_pq.round(3)

In [ ]:
chosen = sel[sel["selected"]]
pool_share = organs.value_counts(normalize=True)
picked_share = chosen["organ"].value_counts(normalize=True)
organ_tab = (
    pd.DataFrame({"udział w puli": pool_share, "udział wśród wybranych": picked_share})
    .fillna(0)
    .sort_values("udział wśród wybranych", ascending=False)
)
organ_tab["średnie PQ w rundzie 2"] = (
    sel[sel["n_labeled"] == 2].groupby("organ")["pq_oracle"].mean()
)
organ_tab.round(3)

In [ ]:
freq = chosen.groupby("image_id")["seed"].nunique().sort_values(ascending=False)
hard = pd.DataFrame(
    {
        "wybrany w ilu seedach (z 6)": freq,
        "narząd": organs.reindex(freq.index),
        "średnie PQ w rundzie 2": sel[sel["n_labeled"] == 2]
        .groupby("image_id")["pq_oracle"]
        .mean()
        .reindex(freq.index),
        "jądra wyroczni": sel.groupby("image_id")["n_oracle"].first().reindex(freq.index),
    }
)
print(f"różnych obrazów wybranych w rundach 2-10: {len(freq)} z {len(POOL)} w puli")
hard.head(15).round(3)

## Koszt adnotacji w jądrach

Obraz to nie jest stała ilość pracy: liczba jąder (według wyroczni) różni się między obrazami kilkukrotnie. Jeśli
podgląd wybiera obrazy z większą liczbą jąder, to przy tej samej liczbie obrazów student dostaje więcej etykiet.

In [ ]:
nuclei = sel.groupby("image_id")[
    "n_oracle"
].first()  # każdy obraz puli jest nieoznaczony w rundzie 2 jakiegoś seeda


def labeled_nuclei(df: pd.DataFrame) -> pd.Series:
    total = [sum(nuclei[i] for i in json.loads(ids)) for ids in df["labeled_ids"]]
    return df.assign(nuclei=total).groupby("n_labeled")["nuclei"].mean()


cost = pd.DataFrame({"losowy": labeled_nuclei(rand), "podgląd": labeled_nuclei(peek)})
cost["podgląd / losowy"] = cost["podgląd"] / cost["losowy"]
print(
    f"obrazów puli z policzonymi jądrami: {len(nuclei)}; średnio {nuclei.mean():.0f} jąder na obraz (min {nuclei.min()}, max {nuclei.max()})"
)
cost.round(2)

## Czy trudność obrazu jest stała

Jeśli te same obrazy są trudne niezależnie od seeda i rundy, to „trudność” jest cechą obrazu (narząd, barwienie), a nie
stanu studenta, i selektor oparty na niepewności będzie w każdym seedzie wybierał podobny zestaw.

In [ ]:
r2 = sel[sel["n_labeled"] == 2].pivot(index="image_id", columns="seed", values="pq_oracle")
rho = r2.corr(method="spearman")
pairs = rho.where(np.triu(np.ones(rho.shape, dtype=bool), k=1)).stack()
print(
    f"runda 2, ranking PQ obrazów puli między seedami: średnia korelacja Spearmana = {pairs.mean():.2f} (min {pairs.min():.2f}, max {pairs.max():.2f})"
)
by_round = sel.pivot_table(
    index="image_id", columns="n_labeled", values="pq_oracle", aggfunc="mean"
)
print(
    "korelacja średniego PQ obrazu między rundą 2 a kolejnymi:",
    {
        int(n): round(by_round[2].corr(by_round[n], method="spearman"), 2)
        for n in by_round.columns
        if n != 2
    },
)

## Czy miary TTA wskazują te same obrazy co podgląd

`uncertainty/pool_scores.py` policzył miary z 8 wersji D4 (`uncertainty/tta.py`) dla studenta każdej rundy, na
nieoznaczonej puli i na walidacji. Miary pochodzą tylko z predykcji studenta; `iou_oracle` i `pq_oracle` służą tu
wyłącznie do sprawdzenia.

**Uwaga o interpretacji.** Sposób agregacji wybieramy na **walidacji**. Pula pokazuje to samo dla obrazów, spośród
których selektor faktycznie wybiera, ale w tym przebiegu podgląd usuwał z niej najtrudniejsze obrazy, więc w późnych
rundach rozrzut PQ w puli jest mały i korelacje są zaniżone.

In [ ]:
RUN = OUT / "al_oracle_peek" / "full"
tta_nuc = pd.concat(
    [pd.read_csv(f) for f in sorted(RUN.glob("seed*/round_*/tta_nuclei.csv"))], ignore_index=True
)
tta_img = pd.concat(
    [pd.read_csv(f) for f in sorted(RUN.glob("seed*/round_*/tta_images.csv"))], ignore_index=True
)
tta_nuc["is_error"] = tta_nuc["iou_oracle"] <= 0.5
print(
    f"jądra: {len(tta_nuc)}, obrazy: {tta_img.groupby('part').size().to_dict()}, "
    f"jądra błędne: {tta_nuc.groupby('part')['is_error'].mean().round(3).to_dict()}"
)


def auc(score, positive) -> float:
    score, positive = np.asarray(score, float), np.asarray(positive, bool)
    ok = ~np.isnan(score)
    score, positive = score[ok], positive[ok]
    n1, n0 = positive.sum(), (~positive).sum()
    return float((stats.rankdata(score)[positive].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


MEASURES = ["K", "U_MI", "U_TTA", "D", "U_H", "Top", "area"]
val_nuc = tta_nuc[tta_nuc["part"] == "val"]
nuc_auc = pd.DataFrame(
    {
        "walidacja": {m: auc(val_nuc[m], val_nuc["is_error"]) for m in MEASURES},
        "pula": {
            m: auc(
                tta_nuc.loc[tta_nuc["part"] == "pool", m],
                tta_nuc.loc[tta_nuc["part"] == "pool", "is_error"],
            )
            for m in MEASURES
        },
    }
)
nuc_auc["siła (walidacja)"] = nuc_auc["walidacja"].map(lambda a: max(a, 1 - a))
nuc_auc.sort_values("siła (walidacja)", ascending=False).round(3)

### Czy `K` to tylko rozmiar jądra

Małe jądra mają wyższe `K` przy tym samym drganiu brzegu i częściej wypadają jako błędne. Jeśli `K` działa także
wewnątrz grup jąder podobnej wielkości, to nie mierzy samego rozmiaru.

In [ ]:
by_size = val_nuc.assign(rozmiar=pd.qcut(val_nuc["area"], 3, labels=["małe", "średnie", "duże"]))
size_tab = by_size.groupby("rozmiar", observed=True).apply(
    lambda g: pd.Series(
        {
            "jądra": len(g),
            "odsetek błędnych": g["is_error"].mean(),
            "AUC K": auc(g["K"], g["is_error"]),
            "AUC U_MI": auc(g["U_MI"], g["is_error"]),
        }
    ),
    include_groups=False,
)
print(
    f"walidacja: AUC samego pola = {max(auc(val_nuc['area'], val_nuc['is_error']), 1 - auc(val_nuc['area'], val_nuc['is_error'])):.3f}, "
    f"korelacja Spearmana K z polem = {val_nuc['K'].corr(val_nuc['area'], method='spearman'):.2f}"
)
print(
    "AUC K w kolejnych rundach (walidacja):",
    {int(n): round(auc(g["K"], g["is_error"]), 3) for n, g in val_nuc.groupby("n_labeled")},
)
size_tab.round(3)

### Poziom obrazu

Dla każdego obrazu składamy miary jąder w jedną liczbę na kilka sposobów i sprawdzamy, w obrębie jednego studenta
(seed, runda), czy ranking obrazów według tej liczby zgadza się z rankingiem według błędu 1 - PQ.

- `rho`: korelacja Spearmana z 1 - PQ, średnia po (seed, runda).
- Tylko pula, rundy 2-10: gdyby selektor wziął 2 obrazy o najwyższym wyniku,
  - `domknięta luka` = jaka część różnicy PQ między średnią puli a wyborem podglądu zostaje osiągnięta
    (0 = jak losowy, 1 = jak podgląd),
  - `wspólne z podglądem` = odsetek wybranych obrazów, które wybrał też podgląd,
  - `wśród 5 najgorszych` = odsetek wybranych obrazów należących do 5 obrazów o najniższym PQ.

In [ ]:
KEY = ["seed", "n_labeled", "image_id"]


def top10(x: pd.Series) -> float:
    v = np.sort(x.dropna().to_numpy())[::-1]
    return v[: max(1, int(np.ceil(0.1 * len(v))))].mean()


grouped = tta_nuc.assign(
    notD=1 - tta_nuc["D"], lowD=tta_nuc["D"] < 1, highK=tta_nuc["K"] > 0.19
).groupby(KEY)
scores = (
    pd.DataFrame(
        {
            "K: średnia": grouped["K"].mean(),
            "K: 10% najgorszych": grouped["K"].apply(top10),
            "K: odsetek > 0,19": grouped["highK"].mean(),
            "U_MI: średnia": grouped["U_MI"].mean(),
            "U_MI: 10% najgorszych": grouped["U_MI"].apply(top10),
            "D: odsetek < 1": grouped["lowD"].mean(),
            "Top: średnia": grouped["Top"].mean(),
        }
    )
    .reset_index()
    .merge(tta_img, on=KEY)
)
scores["E_extra"] = scores["E_extra"]
scores["D: odsetek < 1 + E_extra"] = scores["D: odsetek < 1"] + scores["E_extra"]
scores["liczba jąder"] = scores["n_nuclei"]
scores["u"] = 1 - scores["pq_oracle"]
rank_in_round = scores.groupby(["seed", "n_labeled", "part"])[
    ["K: średnia", "D: odsetek < 1 + E_extra"]
].rank(pct=True)
scores["średnia rang: K + (D, E_extra)"] = rank_in_round.mean(axis=1)
SCORES = [
    "K: średnia",
    "K: 10% najgorszych",
    "K: odsetek > 0,19",
    "U_MI: średnia",
    "U_MI: 10% najgorszych",
    "D: odsetek < 1",
    "E_extra",
    "D: odsetek < 1 + E_extra",
    "Top: średnia",
    "średnia rang: K + (D, E_extra)",
    "liczba jąder",
]


def rho(part: str, col: str) -> pd.Series:
    g = scores[scores["part"] == part].groupby(["seed", "n_labeled"])
    return g.apply(lambda x: x[col].corr(x["u"], method="spearman"), include_groups=False)


def as_selector(col: str) -> pd.Series:
    def one(g: pd.DataFrame) -> pd.Series:
        mine, peeked = g.nlargest(2, col), g.nsmallest(2, "pq_oracle")
        return pd.Series(
            {
                "sel": mine["pq_oracle"].mean(),
                "peek": peeked["pq_oracle"].mean(),
                "pool": g["pq_oracle"].mean(),
                "both": len(set(mine["image_id"]) & set(peeked["image_id"])) / 2,
                "worst5": len(set(mine["image_id"]) & set(g.nsmallest(5, "pq_oracle")["image_id"]))
                / 2,
            }
        )

    pool = scores[(scores["part"] == "pool") & (scores["n_labeled"] <= 10)]
    q = pool.groupby(["seed", "n_labeled"]).apply(one, include_groups=False).mean()
    return pd.Series(
        {
            "domknięta luka": (q["pool"] - q["sel"]) / (q["pool"] - q["peek"]),
            "wspólne z podglądem": q["both"],
            "wśród 5 najgorszych": q["worst5"],
        }
    )


image_tab = pd.DataFrame(
    {
        c: {
            "rho walidacja": rho("val", c).mean(),
            "rho pula": rho("pool", c).mean(),
            "rho pula, runda 2": rho("pool", c).xs(2, level="n_labeled").mean(),
            **as_selector(c),
        }
        for c in SCORES
    }
).T
image_tab = image_tab.sort_values("rho walidacja", ascending=False)
image_tab.to_csv(fig_dir / "tta_image_level.csv")
scores.to_csv(fig_dir / "tta_image_scores.csv", index=False)
image_tab.round(3)

In [ ]:
show = ["K: średnia", "D: odsetek < 1 + E_extra", "U_MI: średnia"]
r2_pool = scores[(scores["part"] == "pool") & (scores["n_labeled"] == 2)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, col in zip(axes, show, strict=True):
    for seed, g in r2_pool.groupby("seed"):
        ax.scatter(g[col], g["pq_oracle"], s=14, alpha=0.7, label=f"seed {seed}")
    ax.set(
        xlabel=col,
        ylabel="PQ obrazu względem wyroczni",
        title=f"runda 2, pula: rho = {-rho('pool', col).xs(2, level='n_labeled').mean():.2f}",
    )
axes[0].legend(fontsize=7)
plt.tight_layout()
fig.savefig(fig_dir / "tta_image_level.png", dpi=130, bbox_inches="tight")

## Selektor `tta_kd` w pętli AL

Ten sam protokół co przebieg z podglądem (krok 2, rundy 2–12, te same seedy i ten sam start), ale selektor nie widzi
masek wyroczni: wynik obrazu to średnia rang `K_mean` i `low_D + E_extra` w puli. Pytanie: jaką część zysku selektora
z podglądem odzyskuje miara liczona bez etykiet.

Dane: `outputs/al_tta_kd/full`.

In [ ]:
tta = load("tta_kd")
t = wide(tta, METRIC)
curves = {"losowy": (r, "tab:gray"), "tta_kd": (t, "tab:blue"), "z podglądem": (p, "tab:red")}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, (w, color) in curves.items():
    axes[0].errorbar(
        w.columns,
        w.mean(),
        yerr=w.std(ddof=1) / np.sqrt(len(w)),
        color=color,
        lw=2,
        marker="o",
        capsize=3,
        label=name,
    )
axes[0].set(
    xlabel="liczba oznaczonych obrazów",
    ylabel="PQ studenta względem wyroczni (walidacja)",
    title="Średnia ± błąd standardowy po seedach",
    xticks=ROUNDS,
)
axes[0].legend()

diff_t = t - r
for seed in diff_t.index:
    axes[1].plot(diff_t.columns, diff_t.loc[seed], marker=".", alpha=0.6, label=f"seed {seed}")
axes[1].plot(
    diff_t.columns, diff_t.mean(), color="tab:blue", lw=2.5, marker="o", label="tta_kd: średnia"
)
axes[1].plot(
    diff.columns, diff.mean(), color="tab:red", lw=2, ls="--", marker="o", label="podgląd: średnia"
)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set(
    xlabel="liczba oznaczonych obrazów",
    ylabel="selektor minus losowy (PQ)",
    title="tta_kd: różnica na seed (porównanie sparowane)",
    xticks=ROUNDS,
)
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
fig.savefig(fig_dir / "tta_kd_vs_random_vs_peek.png", dpi=130, bbox_inches="tight")

In [ ]:
per_round_t = pd.DataFrame({n: paired(diff_t[n]) for n in ROUNDS}).T.rename(
    columns={"seedy z podglądem lepszym": "seedy z tta_kd lepszym"}
)
per_round_t.insert(0, "losowy", r.mean())
per_round_t.insert(1, "tta_kd", t.mean())
per_round_t.insert(2, "podgląd", p.mean())
per_round_t.index.name = "n_labeled"
per_round_t.to_csv(fig_dir / "per_round_tta_kd.csv")
per_round_t

In [ ]:
alc3 = pd.DataFrame({"losowy": alc(r), "tta_kd": alc(t), "podgląd": alc(p)})
alc3["tta_kd − losowy"] = alc3["tta_kd"] - alc3["losowy"]
alc3["podgląd − losowy"] = alc3["podgląd"] - alc3["losowy"]
alc3.to_csv(fig_dir / "alc_tta_kd.csv")
for col in ("tta_kd − losowy", "podgląd − losowy"):
    res = {
        k.replace("seedy z podglądem lepszym", "seedy z dodatnią różnicą"): v
        for k, v in paired(alc3[col]).items()
    }
    print(col, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items()})
gain = alc3[["tta_kd − losowy", "podgląd − losowy"]].mean()
print(f"odzyskana część zysku podglądu (ALC): {gain.iloc[0] / gain.iloc[1]:.0%}")
alc3.round(4)

### Co wybierał `tta_kd`

`pq_oracle_offline` (PQ studenta na obrazie puli względem wyroczni) jest zapisywane tylko do tej analizy; selektor go
nie używa. Dla każdej rundy: korelacja rang wyniku selektora z trudnością obrazu (1 − PQ), średnie PQ dwóch wybranych
obrazów, całej puli i dwóch faktycznie najtrudniejszych (to, co wziąłby podgląd na tej samej puli) oraz rozrzut PQ w puli.

In [ ]:
rows = []
for s in SEEDS:
    for n in ROUNDS[:-1]:
        sel = pd.read_csv(
            OUT / "al_tta_kd" / "full" / f"seed{s}" / f"round_{n:02d}" / "selection.csv"
        )
        rows.append(
            {
                "seed": s,
                "n_labeled": n,
                "rho(wynik, 1 − PQ)": stats.spearmanr(
                    sel["score"], 1 - sel["pq_oracle_offline"]
                ).statistic,
                "PQ wybranych": sel.loc[sel["selected"], "pq_oracle_offline"].mean(),
                "PQ puli": sel["pq_oracle_offline"].mean(),
                "PQ 2 najtrudniejszych": sel["pq_oracle_offline"].nsmallest(2).mean(),
                "odch. std. PQ w puli": sel["pq_oracle_offline"].std(ddof=1),
            }
        )
chosen_t = pd.DataFrame(rows)
chosen_t.groupby("n_labeled").mean().drop(columns="seed").round(3)

In [ ]:
def queried(df: pd.DataFrame) -> dict[int, set[str]]:
    last = df[df["n_labeled"] == ROUNDS[-1]].set_index("seed")["labeled_ids"].map(json.loads)
    return {s: set(ids[ROUNDS[0] :]) for s, ids in last.items()}


q_r, q_p, q_t = queried(rand), queried(peek), queried(tta)
overlap = pd.DataFrame(
    {
        "tta_kd ∩ podgląd": {s: len(q_t[s] & q_p[s]) for s in SEEDS},
        "tta_kd ∩ losowy": {s: len(q_t[s] & q_r[s]) for s in SEEDS},
        "podgląd ∩ losowy": {s: len(q_p[s] & q_r[s]) for s in SEEDS},
    }
)
overlap.index.name = "seed"
print("wspólne obrazy z 10 dobranych w rundach 2-10 (pula ma 28)")
overlap